# ZooMS Clustering Analysis & Results Interpretation
This notebook visualizes and interprets the results of the unsupervised clustering on ZooMS peak lists.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set project paths
RESULTS_DIR = Path("results")

# Load data
metrics_df = pd.read_csv(RESULTS_DIR / "evaluation_metrics.csv", index_col=0)
meta_df = pd.read_csv(RESULTS_DIR / "sample_metadata_phase1.csv").set_index("Sample Name")
cluster_df = pd.read_csv(RESULTS_DIR / "clustering_results.csv", index_col=0)
km_table = pd.read_csv(RESULTS_DIR / "contingency_kmeans.csv", index_col=0)
hier_table = pd.read_csv(RESULTS_DIR / "contingency_hierarchical.csv", index_col=0)

print("Results loaded successfully.")

## 1. Global Performance Metrics
The **Adjusted Rand Index (ARI)** and **V-Measure** give a high-level view of how well the clustering matches the ground truth taxonomy (1.0 = Perfect match).

In [ ]:
display(metrics_df)

metrics_df.plot(kind="bar", figsize=(10, 5))
plt.title("Clustering Performance Metrics")
plt.ylabel("Score")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2. Cluster Purity: Contingency Tables
Heatmaps show which families are successfully isolated into their own clusters and which ones are being mixed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(km_table, annot=True, cmap="YlGnBu", fmt="d", ax=axes[0])
axes[0].set_title("K-Means: Cluster vs. Family Alignment")

sns.heatmap(hier_table, annot=True, cmap="YlGnBu", fmt="d", ax=axes[1])
axes[1].set_title("Hierarchical: Cluster vs. Family Alignment")

plt.tight_layout()
plt.show()

## 3. Dimensionality Reduction Comparison
Re-plotting PCA and t-SNE side-by-side to visually inspect the clusters.

In [ ]:
from src.dim_reduction import DimensionalityReducer
X_binary = pd.read_csv(RESULTS_DIR / "feature_matrix_binary.csv", index_col=0)
reducer = DimensionalityReducer()

pca_df = reducer.run_pca(X_binary)
tsne_df = reducer.run_tsne(X_binary)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.scatterplot(x=pca_df.iloc[:,0], y=pca_df.iloc[:,1], hue=meta_df["Correct ID"], ax=axes[0], palette="tab10", alpha=0.8)
axes[0].set_title("PCA: Colored by Taxonomic Family")
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

sns.scatterplot(x=tsne_df.iloc[:,0], y=tsne_df.iloc[:,1], hue=cluster_df["KMeans_Cluster"], ax=axes[1], palette="tab10", alpha=0.8)
axes[1].set_title("t-SNE: Colored by K-Means Cluster ID")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

## 4. Key Takeaways
1. **Family Separation**: Look at the heatmaps. Families like Bovidae often separate perfectly, while closely related ones might cluster together.
2. **Outliers**: In the scatter plots, look for isolated points. Are they correctly identified in the metadata?
3. **Binary Encoding Results**: With an ARI of ~0.4, there is significant structure being found, but intensity may provide more nuance if we were to switch to the Log-TIC feature set.